# Embeddings Playground — coding workshop (advanced)

**Goes with the Session 1 lesson, Step 2 · Embeddings, after the screen “What we keep: the
embedding table.”** Expected time: ~25 minutes, individual. No GPU needed; the training
runs take a few minutes in total.

In the lesson, skip-gram learned vectors for 8 words from a 12-word corpus, and with 2
dimensions the similar words found each other sooner than with 92. Here you train the same
model, word2vec's skip-gram, on 2 million words of Galician Wikipedia:

- **Part A:** train with 2 to 300 dimensions, and measure time, energy and quality.
- **Part B:** train on less text, and see which words suffer: the lesson's claim that rare
  words get poorly placed vectors.
- **Part C:** test a multilingual model on Galician words that differ from Spanish.

## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried, or run the
   cell under it: that puts the solution into a new code cell for you, ready to run.
   Or tick **fill_in_solutions** in the cell right below these instructions and run that cell:
   from then on, every exercise cell you run gets its solution in place of its code. With the box
   ticked, Runtime → Run all runs the whole notebook with the solutions.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

In [ ]:
# @title Run this cell first. Tick the box to fill in the solutions (point 3 above).
fill_in_solutions = False  # @param {type:"boolean"}

import re

_solutions = {
    1: "def group_scores(wv, k=3):\n    \"\"\"Group score of every test word that has a vector: {word: share of its k nearest test words in its group}.\"\"\"\n    words = [w for w in GROUP_OF if w in wv]\n    V = np.array([wv[w] for w in words])\n    V = V / np.linalg.norm(V, axis=1, keepdims=True)   # unit length: dot product = cosine\n    S = V @ V.T\n    np.fill_diagonal(S, -np.inf)                         # a word is not its own neighbor\n    scores = {}\n    for i, w in enumerate(words):\n        nearest = np.argsort(-S[i])[:k]\n        scores[w] = np.mean([GROUP_OF[words[j]] == GROUP_OF[w] for j in nearest])\n    return scores",
    2: "from gensim.models import Word2Vec\nfrom codecarbon import EmissionsTracker\n\n\ndef get_tracker_energy_kwh(tracker):\n    \"\"\"Total energy of a stopped tracker, in kWh (the attribute moved between CodeCarbon versions).\"\"\"\n    for attr_path in (\"final_emissions_data.energy_consumed\", \"_total_energy.kWh\"):\n        obj = tracker\n        try:\n            for part in attr_path.split(\".\"):\n                obj = getattr(obj, part)\n            return float(obj)\n        except Exception:\n            continue\n    return None\n\n\ndef train(texts, dims):\n    \"\"\"Train skip-gram (sg=1) and measure it: returns (model, seconds, Wh).\"\"\"\n    tracker = EmissionsTracker(measure_power_secs=1, save_to_file=False, log_level=\"error\")\n    tracker.start()\n    t0 = time.perf_counter()\n    model = Word2Vec(texts, vector_size=dims, window=5, min_count=5, sg=1,\n                     negative=5, epochs=5, workers=2, seed=0)\n    seconds = time.perf_counter() - t0\n    tracker.stop()\n    return model, seconds, get_tracker_energy_kwh(tracker) * 1000\n\n\nruns = []\nfor dims in [2, 10, 50, 100, 300]:\n    model, seconds, wh = train(articles, dims)\n    score = np.mean(list(group_scores(model.wv).values()))\n    runs.append((dims, seconds, wh, score))\n    print(f\"{dims:>3} dimensions: {seconds:5.1f} s, {wh:.3f} Wh, group score {score:.2f}\")",
    3: "unit = model.wv.vectors / np.linalg.norm(model.wv.vectors, axis=1, keepdims=True)\n\n\ndef my_most_similar(word, topn=6):\n    q = unit[model.wv.key_to_index[word]]\n    sims = unit @ q\n    best = [i for i in np.argsort(-sims)[:topn + 1] if model.wv.index_to_key[i] != word][:topn]\n    return [(model.wv.index_to_key[i], float(sims[i])) for i in best]\n\n\nfor w in [\"xaneiro\", \"verde\", \"tía\"]:\n    print(w, my_most_similar(w))",
    4: "def softmax_loss(h, out_vectors, ctx):\n    scores = out_vectors @ h\n    scores -= scores.max()\n    return float(-(scores[ctx] - np.log(np.exp(scores).sum())))\n\n\ndef negative_sampling_loss(h, c, negatives):\n    sigmoid = lambda x: 1 / (1 + np.exp(-x))\n    return float(-np.log(sigmoid(c @ h)) - np.log(sigmoid(-(negatives @ h))).sum())\n\n\nrng = np.random.default_rng(0)\ni_in, i_ctx = model.wv.key_to_index[\"xaneiro\"], model.wv.key_to_index[\"febreiro\"]\nh, out = model.wv.vectors[i_in], model.syn1neg\nneg = out[rng.integers(0, len(out), 5)]\nprint(f\"softmax loss {softmax_loss(h, out, i_ctx):.2f}: {len(out):,} dot products\")\nprint(f\"negative-sampling loss {negative_sampling_loss(h, out[i_ctx], neg):.2f}: {1 + len(neg)} dot products\")",
    5: "!pip install -q sentence-transformers\n\nfrom sentence_transformers import SentenceTransformer\n\nmulti_model = SentenceTransformer(\"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2\")\n\nWORDS = [(\"can\", \"perro\", \"dog\"), (\"fiestra\", \"ventana\", \"window\"), (\"cadeira\", \"silla\", \"chair\"),\n         (\"neno\", \"niño\", \"boy\"), (\"irmá\", \"hermana\", \"sister\"), (\"avoa\", \"abuela\", \"grandmother\"),\n         (\"mazá\", \"manzana\", \"apple\"), (\"leite\", \"leche\", \"milk\"), (\"noite\", \"noche\", \"night\"),\n         (\"fillo\", \"hijo\", \"son\"), (\"cidade\", \"ciudad\", \"city\"), (\"auga\", \"agua\", \"water\"),\n         (\"lingua\", \"lengua\", \"language\"),\n         (\"tía\", \"tía\", \"aunt\"), (\"mesa\", \"mesa\", \"table\"), (\"casa\", \"casa\", \"house\"), (\"mar\", \"mar\", \"sea\")]\n\ngl, es, en = (multi_model.encode([t[i] for t in WORDS], normalize_embeddings=True) for i in range(3))\nS_es, S_en = gl @ es.T, gl @ en.T          # cosine similarities: row = Galician word\nS_ee = (es * en).sum(axis=1)                # each Spanish word against its own English translation\n\nright = {\"es\": 0, \"en\": 0}\nprint(f\"{'Galician':>9} | {'nearest Spanish':>20} | {'nearest English':>20} | es–en\")\nfor i, (g, s, e) in enumerate(WORDS):\n    js, je = int(S_es[i].argmax()), int(S_en[i].argmax())\n    right[\"es\"] += js == i\n    right[\"en\"] += je == i\n    mark = lambda ok: \"✓\" if ok else \"✗\"\n    print(f\"{g:>9} | {WORDS[js][1]:>13} {S_es[i, js]:.2f} {mark(js == i)} | {WORDS[je][2]:>13} {S_en[i, je]:.2f} {mark(je == i)} | {S_ee[i]:.2f}\")\nprint(f\"\\nright: Spanish {right['es']} of {len(WORDS)}, English {right['en']} of {len(WORDS)}\")",
}
_fill_in = set(_solutions) if fill_in_solutions else set()


def _fill_in_transformer(lines):
    """An exercise cell starts with `# Exercise N`; with fill-in on for N, it becomes the solution."""
    m = re.match(r"# Exercise (\d+)\s*$", lines[0]) if lines else None
    if not m or int(m.group(1)) not in _fill_in:
        return lines
    n = int(m.group(1))
    return [f"get_ipython().set_next_input(_solutions[{n}], replace=True)\n"] + (_solutions[n] + "\n").splitlines(True)


_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup
                                     if getattr(t, "__name__", "") != "_fill_in_transformer"]
_ip.input_transformers_cleanup.append(_fill_in_transformer)
if fill_in_solutions:
    print("✓ Every exercise cell you run from now on gets its solution.")

## Part A — Train skip-gram on Galician

The text is the Galician sample from Notebook 01b: about 2 million words from random
articles of Galician Wikipedia (CC BY-SA 4.0).

In [ ]:
!pip install -q gensim codecarbon

import gzip
import re
import time
import urllib.request
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://olzama.github.io/teaching/GreenAI/corpora"
WORD = re.compile(r"[^\W\d_]+")      # a word is a run of letters

urllib.request.urlretrieve(f"{DATA_URL}/wiki_gl_2M.txt.gz", "wiki_gl_2M.txt.gz")
with gzip.open("wiki_gl_2M.txt.gz", "rt", encoding="utf-8") as f:
    articles = [WORD.findall(line.lower()) for line in f]
word_counts = Counter(w for a in articles for w in a)
print(f"{len(articles):,} articles, {sum(word_counts.values()):,} words, {len(word_counts):,} different words")

### How to tell whether the vectors are good

Five groups of Galician words. A good embedding puts each word close to the other words of
its group. For every word, take its 3 nearest neighbors among these 44 words; the *group
score* is the share of those neighbors that belong to the same group. 1.0 is perfect.

In [ ]:
GROUPS = {
    "months": "xaneiro febreiro marzo abril maio xuño xullo agosto setembro outubro novembro decembro".split(),
    "days": "luns martes mércores xoves venres sábado domingo".split(),
    "numbers": "dous tres catro cinco seis sete oito nove dez".split(),
    "colors": "vermello azul verde amarelo branco negro".split(),
    "family": "nai pai irmán irmá fillo filla avó avoa tío tía".split(),
}
GROUP_OF = {w: g for g, ws in GROUPS.items() for w in ws}

for g, ws in GROUPS.items():
    print(f"{g:>8}: " + ", ".join(f"{w} {word_counts[w]}" for w in ws))

### ✏️ Exercise 1 — The group score

Complete `group_scores`. For each test word, find its `k` nearest test words by cosine
similarity, and return the share of them in the same group.

The vectors in `V` are already scaled to length 1, so `S = V @ V.T` holds every cosine
similarity, and the diagonal is set to −∞ so a word is never its own neighbor.

<details><summary><b>Hints</b> (click to open)</summary>

- `np.argsort(x)` sorts from smallest to largest: sort `-S[i]` to get the most similar first.
- `GROUP_OF[words[j]] == GROUP_OF[w]` is True when neighbor j is in w's group; the mean of booleans is the share.

</details>

In [ ]:
# Exercise 1
def group_scores(wv, k=3):
    """Group score of every test word that has a vector: {word: share of its k nearest test words in its group}."""
    words = [w for w in GROUP_OF if w in wv]
    V = np.array([wv[w] for w in words])
    V = V / np.linalg.norm(V, axis=1, keepdims=True)   # unit length: dot product = cosine
    S = V @ V.T
    np.fill_diagonal(S, -np.inf)                         # a word is not its own neighbor
    scores = {}
    for i, w in enumerate(words):
        # TODO: the indices of the k most similar test words, then the share in w's group
        ___
    return scores

In [ ]:
# Check for Exercise 1. Run this cell as is.
def test_perfect():
    """two well-separated groups score 1.0"""
    wv = {w: np.array([1.0, 0.05 * i]) for i, w in enumerate(GROUPS["months"][:4])}
    wv.update({w: np.array([0.05 * i, 1.0]) for i, w in enumerate(GROUPS["days"][:4])})
    s = group_scores(wv)
    assert set(s) == set(wv) and all(v == 1.0 for v in s.values())

def test_mixed():
    """a word among the other group scores 0"""
    wv = {"xaneiro": np.array([1.0, 0.0]), "febreiro": np.array([0.95, 0.31]), "luns": np.array([0.0, 1.0])}
    s = group_scores(wv, k=1)
    assert s == {"xaneiro": 1.0, "febreiro": 1.0, "luns": 0.0}, s

def test_missing_words():
    """words without a vector are left out"""
    s = group_scores({"tía": np.array([1.0, 0.0]), "tío": np.array([0.9, 0.1])}, k=1)
    assert set(s) == {"tía", "tío"}

_failed = 0
for _test in [test_perfect, test_mixed, test_missing_words]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
def group_scores(wv, k=3):
    """Group score of every test word that has a vector: {word: share of its k nearest test words in its group}."""
    words = [w for w in GROUP_OF if w in wv]
    V = np.array([wv[w] for w in words])
    V = V / np.linalg.norm(V, axis=1, keepdims=True)   # unit length: dot product = cosine
    S = V @ V.T
    np.fill_diagonal(S, -np.inf)                         # a word is not its own neighbor
    scores = {}
    for i, w in enumerate(words):
        nearest = np.argsort(-S[i])[:k]
        scores[w] = np.mean([GROUP_OF[words[j]] == GROUP_OF[w] for j in nearest])
    return scores
```

</details>

In [ ]:
# Stuck on Exercise 1? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 1" above; your own cell stays as it is.)
get_ipython().set_next_input("def group_scores(wv, k=3):\n    \"\"\"Group score of every test word that has a vector: {word: share of its k nearest test words in its group}.\"\"\"\n    words = [w for w in GROUP_OF if w in wv]\n    V = np.array([wv[w] for w in words])\n    V = V / np.linalg.norm(V, axis=1, keepdims=True)   # unit length: dot product = cosine\n    S = V @ V.T\n    np.fill_diagonal(S, -np.inf)                         # a word is not its own neighbor\n    scores = {}\n    for i, w in enumerate(words):\n        nearest = np.argsort(-S[i])[:k]\n        scores[w] = np.mean([GROUP_OF[words[j]] == GROUP_OF[w] for j in nearest])\n    return scores")


### Train with 2 to 300 dimensions

Skip-gram with a window of 5 words on each side, words that occur at least 5 times, 5
passes over the text. CodeCarbon measures each run.

### ✏️ Exercise 2 — Train skip-gram

Complete the call to `Word2Vec`: skip-gram (`sg=1`; `sg=0` is the other word2vec
model, CBOW), with `dims` dimensions (`vector_size`).

<details><summary><b>Hints</b> (click to open)</summary>

- `vector_size=dims`, `sg=1`.

</details>

In [ ]:
# Exercise 2
from gensim.models import Word2Vec
from codecarbon import EmissionsTracker


def get_tracker_energy_kwh(tracker):
    """Total energy of a stopped tracker, in kWh (the attribute moved between CodeCarbon versions)."""
    for attr_path in ("final_emissions_data.energy_consumed", "_total_energy.kWh"):
        obj = tracker
        try:
            for part in attr_path.split("."):
                obj = getattr(obj, part)
            return float(obj)
        except Exception:
            continue
    return None


def train(texts, dims):
    """Train skip-gram (sg=1) and measure it: returns (model, seconds, Wh)."""
    tracker = EmissionsTracker(measure_power_secs=1, save_to_file=False, log_level="error")
    tracker.start()
    t0 = time.perf_counter()
    # TODO: skip-gram (sg=1) with `dims` dimensions, window 5, min_count 5
    model = Word2Vec(texts, ___,
                     negative=5, epochs=5, workers=2, seed=0)
    seconds = time.perf_counter() - t0
    tracker.stop()
    return model, seconds, get_tracker_energy_kwh(tracker) * 1000


runs = []
for dims in [2, 10, 50, 100, 300]:
    model, seconds, wh = train(articles, dims)
    score = np.mean(list(group_scores(model.wv).values()))
    runs.append((dims, seconds, wh, score))
    print(f"{dims:>3} dimensions: {seconds:5.1f} s, {wh:.3f} Wh, group score {score:.2f}")

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert [r[0] for r in runs] == [2, 10, 50, 100, 300]
assert model.sg == 1, f"the model has sg={model.sg}: pass sg=1 to Word2Vec for skip-gram (the default, sg=0, is CBOW)"
assert model.wv.vector_size == 300, "the last run has 300 dimensions"
assert runs[2][3] > runs[0][3], "50 dimensions should beat 2"
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
from gensim.models import Word2Vec
from codecarbon import EmissionsTracker


def get_tracker_energy_kwh(tracker):
    """Total energy of a stopped tracker, in kWh (the attribute moved between CodeCarbon versions)."""
    for attr_path in ("final_emissions_data.energy_consumed", "_total_energy.kWh"):
        obj = tracker
        try:
            for part in attr_path.split("."):
                obj = getattr(obj, part)
            return float(obj)
        except Exception:
            continue
    return None


def train(texts, dims):
    """Train skip-gram (sg=1) and measure it: returns (model, seconds, Wh)."""
    tracker = EmissionsTracker(measure_power_secs=1, save_to_file=False, log_level="error")
    tracker.start()
    t0 = time.perf_counter()
    model = Word2Vec(texts, vector_size=dims, window=5, min_count=5, sg=1,
                     negative=5, epochs=5, workers=2, seed=0)
    seconds = time.perf_counter() - t0
    tracker.stop()
    return model, seconds, get_tracker_energy_kwh(tracker) * 1000


runs = []
for dims in [2, 10, 50, 100, 300]:
    model, seconds, wh = train(articles, dims)
    score = np.mean(list(group_scores(model.wv).values()))
    runs.append((dims, seconds, wh, score))
    print(f"{dims:>3} dimensions: {seconds:5.1f} s, {wh:.3f} Wh, group score {score:.2f}")
```

</details>

In [ ]:
# Stuck on Exercise 2? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 2" above; your own cell stays as it is.)
get_ipython().set_next_input("from gensim.models import Word2Vec\nfrom codecarbon import EmissionsTracker\n\n\ndef get_tracker_energy_kwh(tracker):\n    \"\"\"Total energy of a stopped tracker, in kWh (the attribute moved between CodeCarbon versions).\"\"\"\n    for attr_path in (\"final_emissions_data.energy_consumed\", \"_total_energy.kWh\"):\n        obj = tracker\n        try:\n            for part in attr_path.split(\".\"):\n                obj = getattr(obj, part)\n            return float(obj)\n        except Exception:\n            continue\n    return None\n\n\ndef train(texts, dims):\n    \"\"\"Train skip-gram (sg=1) and measure it: returns (model, seconds, Wh).\"\"\"\n    tracker = EmissionsTracker(measure_power_secs=1, save_to_file=False, log_level=\"error\")\n    tracker.start()\n    t0 = time.perf_counter()\n    model = Word2Vec(texts, vector_size=dims, window=5, min_count=5, sg=1,\n                     negative=5, epochs=5, workers=2, seed=0)\n    seconds = time.perf_counter() - t0\n    tracker.stop()\n    return model, seconds, get_tracker_energy_kwh(tracker) * 1000\n\n\nruns = []\nfor dims in [2, 10, 50, 100, 300]:\n    model, seconds, wh = train(articles, dims)\n    score = np.mean(list(group_scores(model.wv).values()))\n    runs.append((dims, seconds, wh, score))\n    print(f\"{dims:>3} dimensions: {seconds:5.1f} s, {wh:.3f} Wh, group score {score:.2f}\")")


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot([r[2] for r in runs], [r[3] for r in runs], "o-")
for dims, s, wh, score in runs:
    plt.annotate(f"{dims} dims", (wh, score), textcoords="offset points", xytext=(6, -10))
plt.xlabel("energy of training, Wh")
plt.ylabel("group score")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.show()

**Reflection:** How does the group score change from 2 to 300 dimensions? And the time and
energy? Which number of dimensions would you choose for this corpus, and why? Why might 2
dimensions be too few for 44 words in five groups?

<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

In our run, 2 dimensions gave a group score of 0.71, 10 dimensions 0.93, and 50
dimensions 0.99; 100 and 300 dimensions were no better. Training took 14 seconds with 2
dimensions and 41 with 300, and the energy grew from 0.09 to 0.28 Wh. (The times of the
middle runs are noisy: 50 dimensions happened to take longer than 100.) So past a point,
more dimensions cost more and buy nothing on this test. With 2 dimensions, 44 words in five
groups must share a plane, and the groups overlap. The lesson's toy had the opposite
situation: 8 words, so 2 dimensions were enough. A real system picks the number of
dimensions by testing, as here.

</details>

## Part B — Less text

The lesson said: a rare word takes part in few training pairs, gets few nudges, and ends up
poorly placed. Train with 50 dimensions on the first 100,000 words, then more, and look at
each group separately. The word counts above show which groups are rare.

In [ ]:
def first_words(arts, n_words):
    out, n = [], 0
    for a in arts:
        if n >= n_words:
            break
        out.append(a[:n_words - n])
        n += len(out[-1])
    return out


print(f"{'words':>10} | {'test words with a vector':>24} | " + " | ".join(f"{g:>7}" for g in GROUPS))
for n_words in [100_000, 300_000, 1_000_000, 2_000_000]:
    model, seconds, wh = train(first_words(articles, n_words), 50)
    scores = group_scores(model.wv)
    per_group = [np.mean([scores[w] for w in ws if w in scores]) if any(w in scores for w in ws) else float("nan")
                 for ws in GROUPS.values()]
    print(f"{n_words:>10,} | {len(scores):>24} | " + " | ".join(f"{s:>7.2f}" for s in per_group))

**Reflection:** Which group is good already with 100,000 words, and which is worst? Compare with
the word counts printed in Part A. What does that say about rare words, and about a language
with little text online?

<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

With 100,000 words, the months already score 0.94, but the days of the week
score 0, and only 31 of the 44 test words get a vector at all (a word needs at least 5
occurrences). In all 2 million words, *luns* occurs 16 times and *mércores* 5, against
about 700 for each month. Colors and family words, of middling frequency, score 0.44 and
0.50. With more text, every group improves, and the rarest improve last: at 2 million
words the days reach 0.95. Frequent words are learned from little text; rare words need a
lot. A language with less text online has more words that are rare in its corpus, and so
more poorly placed vectors.

</details>

### Nearest neighbors and a map

The model trained on all 2 million words, with 50 dimensions. Its nearest neighbors come
from the whole vocabulary, not only the 44 test words.

In [ ]:
model, _, _ = train(articles, 50)
for w in ["xaneiro", "verde", "tía"]:
    print(f"{w}: " + ", ".join(f"{n} {s:.2f}" for n, s in model.wv.most_similar(w, topn=6)))

from sklearn.decomposition import PCA

words = [w for w in GROUP_OF if w in model.wv]
coords = PCA(n_components=2).fit_transform(np.array([model.wv[w] for w in words]))
plt.figure(figsize=(7, 6))
for g in GROUPS:
    idx = [i for i, w in enumerate(words) if GROUP_OF[w] == g]
    plt.scatter(coords[idx, 0], coords[idx, 1], label=g)
for (x, y), w in zip(coords, words):
    plt.annotate(w, (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8)
plt.title("50-dimensional Galician skip-gram vectors, projected to 2-D")
plt.legend()
plt.show()

### ✏️ Exercise 3 — Nearest neighbors in one matrix product (advanced)

`most_similar` compares one word with the whole vocabulary. Implement it yourself:
scale every vector of `model.wv.vectors` to length 1 **once**, then one matrix-vector
product gives the cosine of the query with every word. Check against gensim.

<details><summary><b>Hints</b> (click to open)</summary>

- `unit @ q` is one cosine per vocabulary word.
- Sort `-sims`, take `topn + 1` and drop the query word itself.

</details>

In [ ]:
# Exercise 3
unit = model.wv.vectors / np.linalg.norm(model.wv.vectors, axis=1, keepdims=True)


def my_most_similar(word, topn=6):
    q = unit[model.wv.key_to_index[word]]
    # TODO: cosine with every word, then the topn best (leaving out the word itself)
    sims = ___
    best = ___
    return [(model.wv.index_to_key[i], float(sims[i])) for i in best]


for w in ["xaneiro", "verde", "tía"]:
    print(w, my_most_similar(w))

In [ ]:
# Check for Exercise 3. Run this cell as is.
for _w in ["xaneiro", "verde", "tía"]:
    _mine = my_most_similar(_w)
    _ref = model.wv.most_similar(_w, topn=6)
    assert [a for a, _ in _mine] == [a for a, _ in _ref], f"{_w}: {_mine} vs {_ref}"
    assert all(abs(x - y) < 1e-4 for (_, x), (_, y) in zip(_mine, _ref))
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
unit = model.wv.vectors / np.linalg.norm(model.wv.vectors, axis=1, keepdims=True)


def my_most_similar(word, topn=6):
    q = unit[model.wv.key_to_index[word]]
    sims = unit @ q
    best = [i for i in np.argsort(-sims)[:topn + 1] if model.wv.index_to_key[i] != word][:topn]
    return [(model.wv.index_to_key[i], float(sims[i])) for i in best]


for w in ["xaneiro", "verde", "tía"]:
    print(w, my_most_similar(w))
```

</details>

In [ ]:
# Stuck on Exercise 3? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 3" above; your own cell stays as it is.)
get_ipython().set_next_input("unit = model.wv.vectors / np.linalg.norm(model.wv.vectors, axis=1, keepdims=True)\n\n\ndef my_most_similar(word, topn=6):\n    q = unit[model.wv.key_to_index[word]]\n    sims = unit @ q\n    best = [i for i in np.argsort(-sims)[:topn + 1] if model.wv.index_to_key[i] != word][:topn]\n    return [(model.wv.index_to_key[i], float(sims[i])) for i in best]\n\n\nfor w in [\"xaneiro\", \"verde\", \"tía\"]:\n    print(w, my_most_similar(w))")


### ✏️ Exercise 4 — Why skip-gram uses negative sampling (advanced)

The lesson's skip-gram ended in a softmax over the whole vocabulary: one dot product
per word, for every training pair. word2vec replaces it with *negative sampling*: score
the true context word and 5 random words with the sigmoid σ(x) = 1 / (1 + e^(−x)), and
use the loss

L = −log σ(c · h) − Σ log σ(−nᵢ · h)

where h is the input word's vector, c the context word's output vector and nᵢ the 5
negative words' output vectors. Implement both losses for one pair from the trained model
(its output vectors are `model.syn1neg`), and count the dot products each one needs.

<details><summary><b>Hints</b> (click to open)</summary>

- Softmax: −log P(ctx) = −(scores[ctx] − log Σ exp(scores)).
- `negatives @ h` gives the 5 dot products at once; −log σ(x) = log(1 + e^(−x)).
- The count of dot products is the point: the vocabulary size against 6.

</details>

In [ ]:
# Exercise 4
def softmax_loss(h, out_vectors, ctx):
    scores = out_vectors @ h                  # one dot product per vocabulary word
    scores -= scores.max()
    # TODO: −log of the softmax probability of the context word
    return ___


def negative_sampling_loss(h, c, negatives):
    sigmoid = lambda x: 1 / (1 + np.exp(-x))
    # TODO: the loss L above
    return ___


rng = np.random.default_rng(0)
i_in, i_ctx = model.wv.key_to_index["xaneiro"], model.wv.key_to_index["febreiro"]
h, out = model.wv.vectors[i_in], model.syn1neg
neg = out[rng.integers(0, len(out), 5)]
print(f"softmax loss {softmax_loss(h, out, i_ctx):.2f}: {len(out):,} dot products")
print(f"negative-sampling loss {negative_sampling_loss(h, out[i_ctx], neg):.2f}: {1 + len(neg)} dot products")

In [ ]:
# Check for Exercise 4. Run this cell as is.
def test_softmax_small():
    """softmax loss on a 2-word vocabulary"""
    o = np.array([[1.0, 0.0], [0.0, 1.0]])
    h_ = np.array([2.0, 0.0])
    assert abs(softmax_loss(h_, o, 0) - np.log(1 + np.exp(-2.0))) < 1e-9

def test_negative_small():
    """negative sampling on hand values"""
    h_ = np.array([1.0, 0.0])
    L = negative_sampling_loss(h_, np.array([2.0, 0.0]), np.array([[-1.0, 0.0], [0.0, 3.0]]))
    expected = np.log(1 + np.exp(-2.0)) + np.log(1 + np.exp(-1.0)) + np.log(2.0)
    assert abs(L - expected) < 1e-9, f"expected {expected:.4f}, got {L:.4f}"

def test_trained_pair():
    """a true pair of the trained model gets a low loss"""
    assert negative_sampling_loss(h, out[i_ctx], neg) < 1.5

_failed = 0
for _test in [test_softmax_small, test_negative_small, test_trained_pair]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
def softmax_loss(h, out_vectors, ctx):
    scores = out_vectors @ h
    scores -= scores.max()
    return float(-(scores[ctx] - np.log(np.exp(scores).sum())))


def negative_sampling_loss(h, c, negatives):
    sigmoid = lambda x: 1 / (1 + np.exp(-x))
    return float(-np.log(sigmoid(c @ h)) - np.log(sigmoid(-(negatives @ h))).sum())


rng = np.random.default_rng(0)
i_in, i_ctx = model.wv.key_to_index["xaneiro"], model.wv.key_to_index["febreiro"]
h, out = model.wv.vectors[i_in], model.syn1neg
neg = out[rng.integers(0, len(out), 5)]
print(f"softmax loss {softmax_loss(h, out, i_ctx):.2f}: {len(out):,} dot products")
print(f"negative-sampling loss {negative_sampling_loss(h, out[i_ctx], neg):.2f}: {1 + len(neg)} dot products")
```

</details>

In [ ]:
# Stuck on Exercise 4? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 4" above; your own cell stays as it is.)
get_ipython().set_next_input("def softmax_loss(h, out_vectors, ctx):\n    scores = out_vectors @ h\n    scores -= scores.max()\n    return float(-(scores[ctx] - np.log(np.exp(scores).sum())))\n\n\ndef negative_sampling_loss(h, c, negatives):\n    sigmoid = lambda x: 1 / (1 + np.exp(-x))\n    return float(-np.log(sigmoid(c @ h)) - np.log(sigmoid(-(negatives @ h))).sum())\n\n\nrng = np.random.default_rng(0)\ni_in, i_ctx = model.wv.key_to_index[\"xaneiro\"], model.wv.key_to_index[\"febreiro\"]\nh, out = model.wv.vectors[i_in], model.syn1neg\nneg = out[rng.integers(0, len(out), 5)]\nprint(f\"softmax loss {softmax_loss(h, out, i_ctx):.2f}: {len(out):,} dot products\")\nprint(f\"negative-sampling loss {negative_sampling_loss(h, out[i_ctx], neg):.2f}: {1 + len(neg)} dot products\")")


## Part C — A multilingual model and Galician words

`paraphrase-multilingual-MiniLM-L12-v2` is trained on translated sentence pairs in about 50
languages, so translations land close together. Galician is on its model card's list of
languages. Can it match Galician words with their Spanish and English translations?

The test: for each Galician word, find the nearest word among all the Spanish words, and
among all the English words. The first 13 Galician words are spelled differently from
Spanish; the last 4 are spelled the same.

### ✏️ Exercise 5 — Nearest translation

Each row of `S_es` holds the cosine similarities of one Galician word to all the
Spanish words (`S_en`: to the English words). Find the index of the nearest one.

<details><summary><b>Hints</b> (click to open)</summary>

- `S_es[i].argmax()` is the column with the largest similarity in row i.

</details>

In [ ]:
# Exercise 5
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

multi_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

WORDS = [("can", "perro", "dog"), ("fiestra", "ventana", "window"), ("cadeira", "silla", "chair"),
         ("neno", "niño", "boy"), ("irmá", "hermana", "sister"), ("avoa", "abuela", "grandmother"),
         ("mazá", "manzana", "apple"), ("leite", "leche", "milk"), ("noite", "noche", "night"),
         ("fillo", "hijo", "son"), ("cidade", "ciudad", "city"), ("auga", "agua", "water"),
         ("lingua", "lengua", "language"),
         ("tía", "tía", "aunt"), ("mesa", "mesa", "table"), ("casa", "casa", "house"), ("mar", "mar", "sea")]

gl, es, en = (multi_model.encode([t[i] for t in WORDS], normalize_embeddings=True) for i in range(3))
S_es, S_en = gl @ es.T, gl @ en.T          # cosine similarities: row = Galician word
S_ee = (es * en).sum(axis=1)                # each Spanish word against its own English translation

right = {"es": 0, "en": 0}
print(f"{'Galician':>9} | {'nearest Spanish':>20} | {'nearest English':>20} | es–en")
for i, (g, s, e) in enumerate(WORDS):
    # TODO: index of the nearest Spanish word and of the nearest English word
    js, je = ___
    right["es"] += js == i
    right["en"] += je == i
    mark = lambda ok: "✓" if ok else "✗"
    print(f"{g:>9} | {WORDS[js][1]:>13} {S_es[i, js]:.2f} {mark(js == i)} | {WORDS[je][2]:>13} {S_en[i, je]:.2f} {mark(je == i)} | {S_ee[i]:.2f}")
print(f"\nright: Spanish {right['es']} of {len(WORDS)}, English {right['en']} of {len(WORDS)}")

In [ ]:
# Check for Exercise 5. Run this cell as is.
assert right["es"] == sum(int(S_es[i].argmax()) == i for i in range(len(WORDS)))
assert right["en"] == sum(int(S_en[i].argmax()) == i for i in range(len(WORDS)))
assert right["es"] >= 4, "the 4 words spelled like Spanish match themselves"
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

multi_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

WORDS = [("can", "perro", "dog"), ("fiestra", "ventana", "window"), ("cadeira", "silla", "chair"),
         ("neno", "niño", "boy"), ("irmá", "hermana", "sister"), ("avoa", "abuela", "grandmother"),
         ("mazá", "manzana", "apple"), ("leite", "leche", "milk"), ("noite", "noche", "night"),
         ("fillo", "hijo", "son"), ("cidade", "ciudad", "city"), ("auga", "agua", "water"),
         ("lingua", "lengua", "language"),
         ("tía", "tía", "aunt"), ("mesa", "mesa", "table"), ("casa", "casa", "house"), ("mar", "mar", "sea")]

gl, es, en = (multi_model.encode([t[i] for t in WORDS], normalize_embeddings=True) for i in range(3))
S_es, S_en = gl @ es.T, gl @ en.T          # cosine similarities: row = Galician word
S_ee = (es * en).sum(axis=1)                # each Spanish word against its own English translation

right = {"es": 0, "en": 0}
print(f"{'Galician':>9} | {'nearest Spanish':>20} | {'nearest English':>20} | es–en")
for i, (g, s, e) in enumerate(WORDS):
    js, je = int(S_es[i].argmax()), int(S_en[i].argmax())
    right["es"] += js == i
    right["en"] += je == i
    mark = lambda ok: "✓" if ok else "✗"
    print(f"{g:>9} | {WORDS[js][1]:>13} {S_es[i, js]:.2f} {mark(js == i)} | {WORDS[je][2]:>13} {S_en[i, je]:.2f} {mark(je == i)} | {S_ee[i]:.2f}")
print(f"\nright: Spanish {right['es']} of {len(WORDS)}, English {right['en']} of {len(WORDS)}")
```

</details>

In [ ]:
# Stuck on Exercise 5? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 5" above; your own cell stays as it is.)
get_ipython().set_next_input("!pip install -q sentence-transformers\n\nfrom sentence_transformers import SentenceTransformer\n\nmulti_model = SentenceTransformer(\"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2\")\n\nWORDS = [(\"can\", \"perro\", \"dog\"), (\"fiestra\", \"ventana\", \"window\"), (\"cadeira\", \"silla\", \"chair\"),\n         (\"neno\", \"niño\", \"boy\"), (\"irmá\", \"hermana\", \"sister\"), (\"avoa\", \"abuela\", \"grandmother\"),\n         (\"mazá\", \"manzana\", \"apple\"), (\"leite\", \"leche\", \"milk\"), (\"noite\", \"noche\", \"night\"),\n         (\"fillo\", \"hijo\", \"son\"), (\"cidade\", \"ciudad\", \"city\"), (\"auga\", \"agua\", \"water\"),\n         (\"lingua\", \"lengua\", \"language\"),\n         (\"tía\", \"tía\", \"aunt\"), (\"mesa\", \"mesa\", \"table\"), (\"casa\", \"casa\", \"house\"), (\"mar\", \"mar\", \"sea\")]\n\ngl, es, en = (multi_model.encode([t[i] for t in WORDS], normalize_embeddings=True) for i in range(3))\nS_es, S_en = gl @ es.T, gl @ en.T          # cosine similarities: row = Galician word\nS_ee = (es * en).sum(axis=1)                # each Spanish word against its own English translation\n\nright = {\"es\": 0, \"en\": 0}\nprint(f\"{'Galician':>9} | {'nearest Spanish':>20} | {'nearest English':>20} | es–en\")\nfor i, (g, s, e) in enumerate(WORDS):\n    js, je = int(S_es[i].argmax()), int(S_en[i].argmax())\n    right[\"es\"] += js == i\n    right[\"en\"] += je == i\n    mark = lambda ok: \"✓\" if ok else \"✗\"\n    print(f\"{g:>9} | {WORDS[js][1]:>13} {S_es[i, js]:.2f} {mark(js == i)} | {WORDS[je][2]:>13} {S_en[i, je]:.2f} {mark(je == i)} | {S_ee[i]:.2f}\")\nprint(f\"\\nright: Spanish {right['es']} of {len(WORDS)}, English {right['en']} of {len(WORDS)}\")")


**Which side is weak?** In three rows above, the Galician word is close to the English word
but not to the Spanish one. The last column, the Spanish word against its own English
translation, shows the Spanish word is the odd one out there: *leche* is far from *milk* too.
The next cell puts the words into context, and compares them with strings that are words in
no language.

In [ ]:
def sim(a, b):
    va, vb = multi_model.encode([a, b], normalize_embeddings=True)
    return float(va @ vb)

print(f"{'':>26} {'Galician ~ English':>22} {'Spanish ~ English':>24}")
for a, b, c in [("leite", "leche", "milk"), ("o leite", "la leche", "milk"),
                ("Bebo un vaso de leite.", "Bebo un vaso de leche.", "I drink a glass of milk.")]:
    print(f"{c:>26} {a:>17} {sim(a, c):.2f} {b:>19} {sim(b, c):.2f}")

NONSENSE = ["xqzvt", "plsgq", "mynbi"]        # strings that are words in no language
print("\nmean similarity to the nonsense strings")
for w in ["leche", "noche", "neno", "leite", "perro", "milk"]:
    print(f"{w:>6} {sum(sim(w, z) for z in NONSENSE) / len(NONSENSE):.2f}")

**Reflection:** How many Galician words did the model match with Spanish, and with English?
Which words did it get wrong? What do they have in common? Why do the last 4 words get a
similarity of exactly 1.00 with Spanish? In the es–en column, which Spanish words are far
from their own English translation? What does the second cell show about *leche* in context,
and about the words the model gets wrong? Can you tell from these numbers why *leche* alone
is unrecognized? What does all this tell you about a model that saw little Galician? How much does this
model rely on spelling, and would the skip-gram model from Part A rely on it the same way?

<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

In our run the model matched 14 of the 17 Galician words with Spanish, and 14 of
17 with English. Many Galician words are close to their Portuguese or Spanish forms
(*cidade*, *auga*, *avoa*), and the model has seen those languages. It gets these wrong:
*can* (dog) is read as the English word *can*, and *neno* (boy) matches neither *niño* nor
*boy*; both are spelled unlike their Spanish translations (*perro*, *niño*). *irmá* (sister)
lands on Spanish *tía* (aunt), another female relative, and *mar* (sea) matches English *son*. The 4 words spelled like Spanish get a
similarity of exactly 1.00 with Spanish: the model receives the same input and cannot tell
the languages apart.

Some low Galician–Spanish scores come from the Spanish side. *leite*–*leche* is low, but the
es–en column shows that *leche* is far from *milk* as well, while *leite* is right next to
*milk*: the Spanish word is the one out of place. The second cell shows two things. In
context (*la leche*, *un vaso de leche*) the Spanish word is as close to English as the
Galician one. And alone, *leche*, *noche* and *neno* are about as similar to nonsense strings
as to anything else, while *leite*, *perro* and *milk* are not: when the model does not
recognize a bare word, its vector lands in a region where meaningless strings land too. The
model was trained on sentences, and a word by itself is an input it rarely saw. Which bare
words it recognizes cannot be read off the model: the numbers show that *leche* alone is
unrecognized and *leite* alone is not, but not why. Single words out of context are a weak
test of the model.

This model is very attached to spelling. A word spelled like a Spanish or English word gets
that word's vector, whatever language it was meant in: *tía*, *mesa*, *casa* and *mar* get
exactly 1.00 with Spanish, and *can* is taken for the English word. Spelling is not
everything, though: *fiestra* and *cadeira* look nothing like *ventana* and *silla* and
still find them. Not every model works like this. The skip-gram model you trained in Part A
cannot see spelling at all: each word is one row of its embedding table, learned only from
the words around it. To it, *leite* and *leche* are as unrelated as any two words until
their contexts bring them close, and a word it never saw has no vector.

</details>

## Code literacy check

Before you move on, pick at least two code cells above that you actually ran and read
today, and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.